In [11]:
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
import numpy as np
from sklearn.preprocessing import StandardScaler

# Initialization
app = Dash(__name__)

# Read the data
anime = pd.read_csv("../../data/preprocessed_anime.csv")

# Define the numeric columns for correlation
numeric_columns = ['Score', 'Episodes', 'Ranked', 'Popularity', 
                  'Members', 'Favorites', 'Watching', 
                  'Completed', 'On-Hold', 'Dropped']

# reverse rank and popuarity to make it more intuitive, so that the bigger value, means better rank and popularity
anime["Ranked"] = anime["Ranked"].max() - anime["Ranked"]
anime["Popularity"] = anime["Popularity"].max() - anime["Popularity"]

# scale the data first
scaler = StandardScaler()
anime[numeric_columns] = scaler.fit_transform(anime[numeric_columns])

def create_heatmap(df, selected_type=None):
    """
    Create a correlation heatmap based on filtered data
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The anime dataset
    selected_type : str, optional
        The type of anime to filter by (e.g., 'TV', 'Movie', etc.)
    """
    # Filter data if type is selected
    if selected_type and selected_type != 'All':
        df = df[df['Type'] == selected_type]
    
    # Calculate correlation matrix
    correlation_matrix = df[numeric_columns].corr()
    
    # heatmap creation
    fig = go.Figure(data=go.Heatmap(
        z=correlation_matrix,
        x=numeric_columns,
        y=numeric_columns,
        hoverongaps=False,
        # define the max and min for the correlation value
        zmin=-1,
        zmax=1,
        colorscale='RdBu',
        text=np.round(correlation_matrix, 2),
        # correlation value here
        texttemplate='%{text}',
        textfont={"size": 12},
        showscale=True
    ))
    
    # heatmap layout
    fig.update_layout(
        title=f'Anime Correlation Heatmap {f"- {selected_type}" if selected_type else ""}',
        xaxis_title="Metrics",
        yaxis_title="Metrics",
        height=800,
        width=900
    )
    
    return fig

# anime type list for the dropdown
anime_types = ['All']
anime_types.extend(anime['Type'].unique().tolist())

# Define the app layout
app.layout = html.Div([
    html.H1("Anime Correlation Heatmap Dashboard", style={'color': 'white'}),
    html.H3("Catch the unintuitive predictors that affect the popularity", style={'color': 'white'}),
    
    # Dropdown for anime type selection
    html.Div([
        html.Label("Select Anime Type:", style={'color': 'white'}),
        # dropdown for the anime type selection
        dcc.Dropdown(
            id='type-dropdown',
            options=[{'label': type_name, 'value': type_name} 
                    for type_name in anime_types],
            # default value
            value='All',
            style={'width': '200px'}
        )
    ], style={'margin-bottom': '20px'}),
    
    # heatmap graph
    dcc.Graph(id='heatmap-graph')
])

# Callback to update the heatmap based on dropdown selection
@app.callback(
    Output('heatmap-graph', 'figure'),
    [Input('type-dropdown', 'value')]
)
# when the input changes, the heatmap will be updated (callback function)
def update_heatmap(selected_type):
    return create_heatmap(anime, selected_type)

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True) 